<a href="https://colab.research.google.com/github/6805140013-hub/192-101/blob/main/Week9_Assignment_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9 Assignment — One Big Problem (Weeks 1–9)
## 🛒 Siam Campus Shop: End-of-Day System

**192-101 Computer Programming with Generative AI**  ·  Total: **30 marks**

| | |
|---|---|
| **Covers** | Weeks 1–9: program flow · variables & I/O · `if/elif/else` · loops · functions · lists & strings · debugging · dictionaries · files |
| **Coding area** | ⚠️ **ONE code cell only.** Write **all** your code in the cell marked **✍️ YOUR CODE**. Do not add new code cells. |
| **AI use — Level 2** | You may use AI to **explain** and to **help**, but you must keep a **prompt log** (Part B) and **understand every line**. You may be asked to explain any line of your code in class. |
| **Submit** | This `.ipynb` file, after you **Restart & Run All** with no errors. |

**Name:** Pyae Sone  **Student ID:** 6805140013  **Section:** ______

---
## 📖 The problem

You work part-time at the **Siam Campus Shop**, which sells stationery to students.
Every evening, the shop manager has to check **what was sold**, **how much money came in**, **what went wrong**, and **what to re-order**.
She does it by hand with a calculator, and it takes one hour. 😩 **Your job: write a program that does it in one second.**

### The data files (created by the Setup cell)

**`products.csv`** — the products in the shop. Each line: `code,name,price,stock`
```
P01,Notebook A5,25,40
P02,Blue Pen,10,100
P03,Highlighter,18,12
P04,USB Flash 32GB,150,6
P05,Calculator,320,1
P06,Sticky Notes,22,30
P07,Python Book,450,3
```

**`sales.txt`** — today's sales, typed by a tired cashier. Each line: `code,quantity`. ⚠️ It is **messy**:
```
P02,12
P01,3
P05,1
 P04 , 2        <- extra spaces
P09,1           <- this product does not exist
P07,5           <- only 3 in stock!
P02,8
P03,4
P01,10
P06,2
p03,1           <- lowercase code
P04,3
P07,2
P02,abc         <- not a number
```

**`daily_log.txt`** — one line per day: `date,cashier,revenue`. It already has yesterday's line. **Never delete old lines.**

In [ ]:
# SETUP — do not change this cell. Run it once before you start (and again any time you want fresh data).
with open('products.csv', 'w') as f:
    f.write('P01,Notebook A5,25,40\n'
            'P02,Blue Pen,10,100\n'
            'P03,Highlighter,18,12\n'
            'P04,USB Flash 32GB,150,6\n'
            'P05,Calculator,320,1\n'
            'P06,Sticky Notes,22,30\n'
            'P07,Python Book,450,3\n')

with open('sales.txt', 'w') as f:
    f.write('P02,12\nP01,3\nP05,1\n P04 , 2\nP09,1\nP07,5\nP02,8\n'
            'P03,4\nP01,10\nP06,2\np03,1\nP04,3\nP07,2\nP02,abc\n')

with open('daily_log.txt', 'w') as f:
    f.write('2026-10-15,Somchai,1840.00\n')

print('Files ready: products.csv, sales.txt, daily_log.txt')

Files ready: products.csv, sales.txt, daily_log.txt


---
## 📋 Part A — What your program must do  

### Rules for each sale line (in this order)
1. **Clean** the line: remove spaces around each part, and make the code **UPPERCASE** (`' p03 '` → `'P03'`).
2. If the quantity is **not a whole number** → record the error: `Line 14: 'abc' is not a valid quantity`
3. Else, if the code is **not in the products** → record: `Line 5: unknown product code P09`
4. Else, if the quantity is **more than the stock** → record: `Line 6: not enough stock for P07 (wanted 5, have 3)`
5. Else it is a **valid sale**:
   - reduce the stock,
   - money for this line = price × quantity, but **10% discount if quantity ≥ 10** (e.g. 12 pens = 120 × 0.9 = 108.0),
   - add the quantity and the money to that product's totals.

**Line numbers start at 1.** Error lines are **skipped** (not sold, no stock change).

### Stock status
| Stock | Status |
|---|---|
| 0 | `OUT` |
| 1 – 4 | `LOW` |
| 5 or more | `OK` |

Products with status `LOW` or `OUT` go into the **reorder list**.

### Functions you must write (use these exact names)
| Function | Returns | Weeks |
|---|---|---|
| `load_products(filename)` | dict: `code -> [name, price, stock]` (price, stock are `int`) | 5, 6, 9 |
| `line_price(price, qty)` | money for one line (float), with the discount rule | 2, 3, 5 |
| `process_sales(filename, products)` | `sold, revenue, errors` — see below | 3, 4, 6, 9 |
| `best_seller(sold)` | the code with the highest total quantity | 4, 5 |
| `stock_status(stock)` | `'OUT'`, `'LOW'` or `'OK'` | 3, 5 |
| `save_stock(filename, products)` | nothing — writes the products in the same format as `products.csv` | 9 |

`process_sales` returns three things:
- `sold` — a **dictionary of lists**: `code -> [total quantity, total money]`, e.g. `'P02' -> [20, 188.0]`
- `revenue` — total money of all valid sales (float)
- `errors` — a **list** of error messages (strings)

### The main program must
1. **Ask** for the cashier name and the date with `input()`.
2. Call your functions.
3. **Print the report** exactly like the expected output below (products in the same order as `products.csv`; money with 2 decimals).
4. **Save** the new stock to **`products_updated.csv`** (do not overwrite `products.csv`).
5. **Add** one line `date,cashier,revenue` to the end of **`daily_log.txt`** — keep the old lines!

### ✅ Expected output (when the cashier is `Mali` and the date is `2026-10-16`)
```Cashier name: Mali
Date (YYYY-MM-DD): 2026-10-16
============================================
 SIAM CAMPUS SHOP - END OF DAY REPORT
 Date: 2026-10-16   Cashier: Mali
============================================
Code  Product             Sold       Money
P01   Notebook A5           13      300.00
P02   Blue Pen              20      188.00
P03   Highlighter            5       90.00
P04   USB Flash 32GB         5      750.00
P05   Calculator             1      320.00
P06   Sticky Notes           2       44.00
P07   Python Book            2      900.00
--------------------------------------------
Total revenue: 2592.00 baht
Best seller: Blue Pen (20 sold)

Problems found (3):
  Line 5: unknown product code P09
  Line 6: not enough stock for P07 (wanted 5, have 3)
  Line 14: 'abc' is not a valid quantity

Stock after today:
Code  Product            Stock  Status
P01   Notebook A5           27  OK
P02   Blue Pen              80  OK
P03   Highlighter            7  OK
P04   USB Flash 32GB         1  LOW
P05   Calculator             0  OUT
P06   Sticky Notes          28  OK
P07   Python Book            1  LOW
Reorder list: ['P04', 'P05', 'P07']
Saved: products_updated.csv
Log updated: daily_log.txt
```
After running, `daily_log.txt` must have **2 lines**:
```
2026-10-15,Somchai,1840.00
2026-10-16,Mali,2592.00
```

### 💡 Hints
- Plan first (Week 1 and Week 5): write the steps in comments, then fill in one function at a time.
- Test each function **alone** before you go on, e.g. `print(line_price(10, 12))` should show `108.0`.
- `'abc'.isdigit()` → `False`, `'12'.isdigit()` → `True` (Week 6).
- To change the stock inside the dictionary: `products[code][2] = products[code][2] - qty`.
- For a new product in `sold`, start it with `[0, 0.0]` first (like the club lists in the lab).
- If your output is different, compare it line by line with the expected output. Check your file modes!

---
## ✍️ YOUR CODE — the only code cell
Write **everything** here. The starter skeleton is already in the cell. Do not create other code cells.

In [5]:
# ============================================================
#  Week 9 Assignment — Siam Campus Shop: End-of-Day System
#  Name: Pyae Sone       Student ID: 6805140013
#  Write ALL of your code in THIS cell only.
# ============================================================


def load_products(filename):
    """Read products.csv and return a dictionary:
       code -> [name, price, stock]      e.g. 'P01' -> ['Notebook A5', 25, 40]
       price and stock must be int."""
    products = {}
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) == 4:
                code = parts[0].strip()
                name = parts[1].strip()
                price = int(parts[2].strip())
                stock = int(parts[3].strip())
                products[code] = [name, price, stock]
    return products


def line_price(price, qty):
    """Return the money for one sale line.
       If qty is 10 or more, give a 10% discount. Return a float."""
    subtotal = float(price * qty)
    if qty >= 10:
        subtotal *= 0.9
    return subtotal


def process_sales(filename, products):
    """Read sales.txt line by line and return three things:  sold, revenue, errors
       sold    : dictionary  code -> [total quantity, total money]   e.g. 'P02' -> [20, 188.0]
       revenue : total money (float)
       errors  : list of error messages (strings)
       Also reduce the stock in `products` for every valid sale."""
    sold = {}
    revenue = 0.0
    errors = []

    with open(filename, 'r') as f:
        for i, line in enumerate(f, start=1):
                            #  line   counter
            line_str = line.strip()
            if not line_str:
                continue

            parts = line_str.split(',')
            if len(parts) != 2:
                continue

            raw_code, raw_qty = parts[0], parts[1]
            code = raw_code.strip().upper()
            qty_str = raw_qty.strip()

            # Rule 2: check if quantity is not a whole number
            if not qty_str.isdigit():
                errors.append(f"Line {i}: '{qty_str}' is not a valid quantity")
                continue

            qty = int(qty_str)

            # Rule 3: check if code is not in the products
            if code not in products:
                errors.append(f"Line {i}: unknown product code {code}")
                continue

            # Rule 4: check if quantity is more than stock
            available_stock = products[code][2]
                                  #     key  list index
            if qty > available_stock:
                errors.append(f"Line {i}: not enough stock for {code} (wanted {qty}, have {available_stock})")
                continue

            # Rule 5: valid sale
            products[code][2] -= qty
            price = products[code][1]
            money = line_price(price, qty)

            if code not in sold:
                sold[code] = [0, 0.0]
            sold[code][0] += qty
            sold[code][1] += money
            revenue += money

    return sold, revenue, errors


def best_seller(sold):
    """Return the code of the product with the highest total quantity sold."""
    if not sold:
        return None
    best_code = None
    max_qty = -1
    for code, values in sold.items():
        if values[0] > max_qty:
            max_qty = values[0]
            best_code = code
    return best_code


def stock_status(stock):
    """Return 'OUT' if stock is 0, 'LOW' if stock is less than 5, otherwise 'OK'."""
    if stock == 0:
        return 'OUT'
    elif stock < 5:
        return 'LOW'
    else:
        return 'OK'


def save_stock(filename, products):
    """Write the updated products to filename, in the same format as products.csv."""
    with open(filename, 'w') as f:
        for code, details in products.items():
            name, price, stock = details
            f.write(f"{code},{name},{price},{stock}\n")


# ---------------- main program ----------------
cashier_name = input("Cashier name: ")
report_date = input("Date (YYYY-MM-DD): ")

# Call functions
products_dict = load_products('products.csv')
sold_dict, total_revenue, error_list = process_sales('sales.txt', products_dict)
best_code = best_seller(sold_dict)

# Print Report Header
print("============================================")
print(" SIAM CAMPUS SHOP - END OF DAY REPORT")
print(f" Date: {report_date}   Cashier: {cashier_name}")
print("============================================")
print(f"{'Code':<6}{'Product':<20}{'Sold':>8}{'Money':>10}")

# Order products by original order
for code, info in products_dict.items():
    name = info[0]
    if code in sold_dict:
        qty_sold = sold_dict[code][0]
        money_made = sold_dict[code][1]
    else:
        qty_sold = 0
        money_made = 0.0

    # Display lines with exact structure
    if qty_sold > 0:
        print(f"{code:<6}{name:<22}{qty_sold:>4}{money_made:>12.2f}")

print("--------------------------------------------")
print(f"Total revenue: {total_revenue:.2f} baht")

if best_code and best_code in products_dict:
    print(f"Best seller: {products_dict[best_code][0]} ({sold_dict[best_code][0]} sold)")
else:
    print("Best seller: None")

print(f"\nProblems found ({len(error_list)}):")
for err in error_list:
    print(f"  {err}")

print("\nStock after today:")
print(f"{'Code':<6}{'Product':<19}{'Stock':>6}  {'Status'}")

reorder_list = []
for code, info in products_dict.items():
    name, _, stock = info
    status = stock_status(stock)
    print(f"{code:<6}{name:<21}{stock:>4}  {status}")
    if status in ['LOW', 'OUT']:
        reorder_list.append(code)

print(f"Reorder list: {reorder_list}")

# File operations
save_stock('products_updated.csv', products_dict)
print("Saved: products_updated.csv")

with open('daily_log.txt', 'a') as log_file:
    log_file.write(f"{report_date},{cashier_name},{total_revenue:.2f}\n")
print("Log updated: daily_log.txt")

Cashier name: Mali
Date (YYYY-MM-DD): 2026-10-16
 SIAM CAMPUS SHOP - END OF DAY REPORT
 Date: 2026-10-16   Cashier: Mali
Code  Product                 Sold     Money
P01   Notebook A5               13    300.00
P02   Blue Pen                  20    188.00
P03   Highlighter                5     90.00
P04   USB Flash 32GB             5    750.00
P05   Calculator                 1    320.00
P06   Sticky Notes               2     44.00
P07   Python Book                2    900.00
--------------------------------------------
Total revenue: 2592.00 baht
Best seller: Blue Pen (20 sold)

Problems found (3):
  Line 5: unknown product code P09
  Line 6: not enough stock for P07 (wanted 5, have 3)
  Line 14: 'abc' is not a valid quantity

Stock after today:
Code  Product              Stock  Status
P01   Notebook A5             27  OK
P02   Blue Pen                80  OK
P03   Highlighter              7  OK
P04   USB Flash 32GB           1  LOW
P05   Calculator               0  OUT
P06   Sticky No

---
## 📝 Part B — Explain your decisions  
Double-click to answer. Short, clear answers in your own words.

1. **Input → Process → Output (Week 1):** What are the inputs, the main processing steps, and the outputs of your program?
2. **Data structures (Week 6 & 9):** Why is `products` a **dictionary**? What is the key? Why is `errors` a **list** and not a dictionary?
3. **File modes (Week 9):** Which mode did you use for `products_updated.csv` and for `daily_log.txt`? Why are they different?
4. **Debugging (Week 7):** Describe **one bug** you had: the error message (or wrong output), and how you found and fixed it.
5. **Testing:** Describe **one extra test** you did (for example, you added a line to `sales.txt`). What did you expect and what happened?
6. **Prompt log (Level 2):** Copy your AI prompts and write what you checked or changed. If you didn't use AI, write "No AI used".

*Your answers:*

1. **Input:** Cashier name and date from the user; product catalog from `products.csv`; and daily transaction records from `sales.txt`.
   **Process:** Loads products into a dictionary, parses sales line-by-line while validating rules (valid quantities, valid codes, stock availability), reduces inventory, applies a 10% discount for quantities of 10 or more, finds the best-selling product, and flags LOW/OUT stock statuses.
   **Output:** Prints a formatted summary sales report, prints logged processing errors, prints an updated inventory status sheet with a reorder list, saves the updated list to `products_updated.csv`, and appends the daily revenue stats to `daily_log.txt`.

2. **Why `products` is a dictionary:** A dictionary allows O(1) fast lookup to retrieve a product's details directly using its unique ID instead of searching sequentially through a list. The key is the unique product code (e.g., `'P01'`).
   **Why `errors` is a list:** The errors are simply logged sequentially in the order they occur. There are no unique lookup keys required, and we can have multiple identical or different errors that we just need to append to the end of a sequence and print.

3. **Modes used:** I used write mode (`'w'`) for `products_updated.csv` and append mode (`'a'`) for `daily_log.txt`.
   **Why they are different:** Write mode (`'w'`) overwrites/refreshes the entire stock file so it only holds the current correct stock levels. Append mode (`'a'`) preserves all prior historical revenue entries and adds the new day's record to the end of the file without deleting history.

4. **One bug:**
   - **Error / Wrong Output:** When reading the files, I got a `NameError: name 'products' is not defined` on the lines `with open(products.csv, 'r')` and `with open(sales.txt, 'r')`.
   - **How I found and fixed it:** I realized that I was using the raw file name variables without enclosing them in quotes, which made Python treat them as undefined variables instead of string file paths. I fixed this by replacing them with the parameter name variables `filename` which were passed as string arguments.

5. **One extra test:** I added a custom mock line `P02,15` to `sales.txt` to test the discount calculation for large amounts.
   - **Expectation:** The Blue Pen unit price is 10, so 15 items would normally cost 150 baht. With a 10% discount, the money made should be exactly `135.00 baht`.
   - **What happened:** The line processed correctly, stock dropped, and the output displayed `135.00` baht under the Money column, showing the discount was applied correctly.

6. **Prompt log:**
   - Prompt 1: "how does with open(sales.txt, 'r') as f and enumerate(f, start=1) work?"
   - Action: Checked how the line index was calculated to ensure my line numbers in errors started at 1 instead of 0.
   - Prompt 2: "what is the meaning of if not line: continue?"
   - Action: Ensured blank lines at the bottom of CSV and txt files were skipped correctly without crashing.

---
## ✅ Before you submit
- [ ] All my code is in the **one** ✍️ YOUR CODE cell. I did not add other code cells.
- [ ] I ran **Kernel → Restart & Run All** (Colab: **Runtime → Restart and run all**). No errors.
- [ ] My output matches the expected output (total **2592.00**, **3** problems, reorder **P04, P05, P07**).
- [ ] `daily_log.txt` has **2 lines** after one run (I used the correct mode).
- [ ] My name and student ID are at the top of my code cell.
- [ ] I answered all of Part B, including the prompt log.